In [38]:
import pandas as pd
import statsmodels.formula.api as smf
import scipy.stats as st
import matplotlib.pyplot as plt
from statsmodels.iolib.summary2 import summary_col as sc
import numpy as np
from scipy.stats import sem

url = 'https://raw.githubusercontent.com/git-iush/large_family/refs/heads/main/data2.parquet'

In [11]:
df = pd.read_parquet(url)
df.head()

,SERIAL,HHWT,CLUSTER,STRATA,PERNUM,PERWT,NCHILD,SEX,AGE,MARST,...,BLACK,AIAN,API,OTHER,MULTR,HISP,EXPANSION,WELFARE,PRSNINC,YRSEDUC
0,4,4.0,2024000000041,280201,1,4.0,0,2,22,6,...,0,0,1,0,0,0,0,0.0,1000.0,13
1,9,74.0,2024000000091,280301,1,74.0,0,2,18,6,...,1,0,0,0,0,0,0,0.0,5100.0,12
2,12,55.0,2024000000121,160001,1,55.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,0.0,12
3,15,17.0,2024000000151,120201,1,17.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,6000.0,12
4,19,71.0,2024000000191,10001,1,71.0,0,2,23,6,...,0,0,0,0,0,0,0,0.0,0.0,13


In [35]:
states = 'DC + ALABAMA + ALASKA + ARIZONA + ARKANSAS + CALIFORNIA + COLORADO + CONNETICUT + DELAWARE + GEORGIA + HAWAII + IDAHO + ILLINOIS + INDIANA + IOWA + KANSAS + KENTUCKY + LOUISIANA + MAINE + MARYLAND + MASSACHUSETTS + MICHIGAN + MINNESOTA + MISSISSIPPI + MISSOURI + MONTANA + NEBRASKA + NEVADA + NEWHAMPSHIRE + NEWJERSEY + NEWMEXICO + NEWYORK + NORTHCAROLINA + NORTHDAKOTA + OHIO + OKLAHOMA + OREGON + PENNSYLVANIA + RHODEISLAND + SOUTHCAROLINA + SOUTHDAKOTA + TENNESSEE + TEXAS + UTAH + VERMONT + VIRGINIA + WASHINGTON + WESTVIRGINIA + WISCONSIN + WYOMING'
races = 'BLACK + AIAN + API + OTHER + MULTR + HISP'
educs = 'HIGHSCHOOL + BACHELORS + ADVANCED'

In [13]:
df['MEDICAID'] = df['HINSCAID'] - 1
df['MEDFARE'] = df['MEDICAID'] * df['WELFARE']

df['MEDICAGE'] = df['MEDICAID'] * df['AGE']
df['WELFAGE'] = df['WELFARE'] * df['AGE']
df['MEDFARAGE'] = df['MEDFARE'] * df['AGE']

df['MEDICAGE2'] = df['MEDICAGE'] ** 2
df['WELFAGE2'] = df['WELFAGE'] ** 2 
df['MEDFARAGE2'] = df['MEDFARAGE'] ** 2

df['FAMINC'] = df['FTOTINC'] - df['INCTOT']
df['PRSNINC2'] = df['PRSNINC'] ** 2
df['FAMINC2'] = df['FAMINC'] ** 2

df['AGE2'] = df['AGE'] ** 2

df['HIGHSCHOOL'] = [1 if i == 63 or i==64 else 0 for i in df['EDUCD']]
df['BACHELORS'] = [1 if i == 101 else 0 for i in df['EDUCD']]
df['ADVANCED'] = [1 if i>=114 else 0 for i in df['EDUCD']]

df['LARGEFAM'] = [1 if i >= 3 else 0 for i in df['NCHILD']]

df['MARD'] = [1 if i==1 or i==2 or i==3 else 0 for i in df['MARST']]

In [14]:
log = f'LARGEFAM ~ MEDICAID + WELFARE + MEDFARE + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARD + {races} + {educs} + {states}'
logres = smf.logit(log, data=df).fit()

lex = f'LARGEFAM ~ MEDICAID + WELFARE + MEDFARE + EXPANSION + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARD + {races} + {educs}'
lexres = smf.logit(lex,data=df).fit()

Optimization terminated successfully.
         Current function value: 0.282860
         Iterations 9


In [15]:
print(logres.summary().tables[1])
print(logres.get_margeff(method='dydx',at='mean',dummy=True).summary())

                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept       -19.9450      0.127   -157.587      0.000     -20.193     -19.697
MEDICAID          0.7725      0.012     65.945      0.000       0.750       0.795
WELFARE          -0.3654      0.094     -3.870      0.000      -0.550      -0.180
MEDFARE           0.5817      0.099      5.905      0.000       0.389       0.775
AGE               0.8949      0.007    134.131      0.000       0.882       0.908
AGE2             -0.0114   8.75e-05   -129.954      0.000      -0.012      -0.011
PRSNINC       -6.159e-06   1.33e-07    -46.240      0.000   -6.42e-06    -5.9e-06
PRSNINC2       8.481e-12   2.42e-13     35.040      0.000    8.01e-12    8.96e-12
FAMINC         2.605e-06   8.88e-08     29.351      0.000    2.43e-06    2.78e-06
FAMINC2       -2.268e-12   1.38e-13    -16.430      0.000   -2.54e-12      -2e-12
MARD            

In [32]:
print(lexres.summary().tables[1])
print(lexres.get_margeff(method='dydx',at='mean',dummy=True).summary())

Optimization terminated successfully.
         Current function value: 0.284784
         Iterations 9
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -19.4055      0.125   -155.766      0.000     -19.650     -19.161
MEDICAID       0.7283      0.012     62.988      0.000       0.706       0.751
WELFARE       -0.4108      0.094     -4.362      0.000      -0.595      -0.226
MEDFARE        0.6158      0.098      6.265      0.000       0.423       0.808
EXPANSION     -0.0245      0.009     -2.592      0.010      -0.043      -0.006
AGE            0.8885      0.007    133.622      0.000       0.875       0.902
AGE2          -0.0113   8.72e-05   -129.515      0.000      -0.011      -0.011
PRSNINC    -6.471e-06   1.32e-07    -49.032      0.000   -6.73e-06   -6.21e-06
PRSNINC2    8.745e-12    2.4e-13     36.366      0.000    8.27e-12    9.22e-12
FAMINC      2.431e-06   8.76e

In [36]:
df['yhat_log'] = pd.Series(logres.predict(df), index=logres.model.data.row_labels)
df['yhat_lex'] = pd.Series(lexres.predict(df), index=lexres.model.data.row_labels)
rss_logit = (df['NCHILD'] - df['yhat_log']).sum() ** 2

In [ ]:
cm_log = pd.crosstab(df['LARGEFAM'],df['yhat_log'],margins=True)
cm_lex = pd.crosstab(df['LARGEFAM',df['yhat_lex'],margins=True)

In [ ]:
print(cm_log)
print(cm_lex)